# Stage 2: Relational Data Modeling & Analytical SQL

**Objective:** Ingest processed Spotify streaming data into a relational SQLite database, normalize dimensions, and run business-focused analytical queries using SQL.

In [9]:
import sqlite3
import pandas as pd

# Connect and initialize local relational SQLite database
conn = sqlite3.connect('../data/spotify_warehouse.db')
cursor = conn.cursor()

# Load processed dataset from Stage 1
df = pd.read_csv('../data/spotify_data_processed.csv')

# Load raw staging data into relational storage
df.to_sql('staging_tracks', conn, if_exists='replace', index=False)

print(f"Staging table created with {len(df):,} records.")

Staging table created with 85,000 records.


## 1. Schema Normalization (Star Schema / 3NF)

I separate the large flat table into distinct dimensions (`dim_labels`, `dim_genres`) and a central metrics table (`fact_tracks`). This eliminates repeated text values, ensures data integrity, and improves query speed via indexes.

In [10]:
# Create normalized dimension and fact tables
cursor.executescript("""
DROP TABLE IF EXISTS dim_labels;
DROP TABLE IF EXISTS dim_genres;
DROP TABLE IF EXISTS fact_tracks;

-- Dimension 1: Record Labels
CREATE TABLE dim_labels (
    label_id INTEGER PRIMARY KEY AUTOINCREMENT,
    label_name TEXT UNIQUE NOT NULL
);

-- Dimension 2: Music Genres
CREATE TABLE dim_genres (
    genre_id INTEGER PRIMARY KEY AUTOINCREMENT,
    genre_name TEXT UNIQUE NOT NULL
);

-- Fact Table: Tracks with metrics and foreign keys
CREATE TABLE fact_tracks (
    track_id TEXT PRIMARY KEY,
    track_name TEXT,
    artist_name TEXT,
    label_id INTEGER,
    genre_id INTEGER,
    popularity INTEGER,
    stream_count REAL,
    danceability REAL,
    energy REAL,
    loudness REAL,
    tempo REAL,
    release_year INTEGER,
    is_explicit INTEGER,
    FOREIGN KEY (label_id) REFERENCES dim_labels(label_id),
    FOREIGN KEY (genre_id) REFERENCES dim_genres(genre_id)
    
);

-- Populate Dimensions with unique values
INSERT OR IGNORE INTO dim_labels (label_name)
SELECT DISTINCT label FROM staging_tracks WHERE label IS NOT NULL;

INSERT OR IGNORE INTO dim_genres (genre_name)
SELECT DISTINCT label FROM staging_tracks WHERE genre IS NOT NULL;

-- Populate Fact Table using foreign keys from dimensions
INSERT OR IGNORE INTO fact_tracks
SELECT
    s.track_id,
    s.track_name,
    s.artist_name,
    l.label_id,
    g.genre_id,
    s.popularity,
    s.stream_count,
    s.danceability,
    s.energy,
    s.loudness,
    s.tempo,
    s.release_year,
    s.is_explicit_bool
FROM staging_tracks s
LEFT JOIN dim_labels l ON s.label = l.label_name
LEFT JOIN dim_genres g ON s.genre = g.genre_name;

-- Indexes to make analytics queries fast
CREATE INDEX idx_tracks_popularity ON fact_tracks(popularity);
CREATE INDEX idx_tracks_genre ON fact_tracks(genre_id);
""")

conn.commit()
print("Relational schema and indexes created successfully!")

Relational schema and indexes created successfully!


## 2. Business Analysis 1: Top 3 Most Streamed Tracks per Genre

**Business Scenario:** The content curation team needs to identify the top 3 most-streamed tracks for each genre to build featured playlists. We use a Window Function (`ROW_NUMBER()`) to rank tracks within each category.

In [11]:
query_1 = """
WITH ranked_tracks AS (
    SELECT
        g.genre_name,
        t.track_name,
        t.artist_name,
        t.stream_count,
        t.popularity,
        ROW_NUMBER() OVER(
            PARTITION BY g.genre_name
            ORDER BY t.stream_count DESC
        ) AS genre_rank
    FROM fact_tracks t
    JOIN dim_genres g ON t.genre_id = g.genre_id
)
SELECT
    genre_name,
    genre_rank,
    track_name,
    artist_name,
    ROUND(stream_count, 0) AS total_streams,
    popularity
FROM ranked_tracks
WHERE genre_rank <= 3
ORDER BY genre_name, genre_rank
LIMIT 12;
"""

df_top_genre = pd.read_sql_query(query_1, conn)
df_top_genre

,genre_name,genre_rank,track_name,artist_name,total_streams,popularity


## 3. Business Analysis 2: Record Label Hit Density & Catalog Performance

**Business Scenario:** Management wants to evaluate which record labels produce the highest percentage of hit songs (Popularity $\ge$ 75) relative to their catalog size (minimum 100 tracks).

In [12]:
query_2 = """
SELECT 
    l.label_name,
    COUNT(t.track_id) AS total_tracks,
    SUM(CASE WHEN t.popularity >= 75 THEN 1 ELSE 0 END) AS hit_tracks,
    ROUND(AVG(t.popularity), 1) AS avg_popularity,
    ROUND(SUM(t.stream_count), 0) AS total_streams,
    ROUND((CAST(SUM(CASE WHEN t.popularity >= 75 THEN 1 ELSE 0 END) AS REAL) / COUNT(t.track_id)) * 100, 2) AS hit_ratio_pct
FROM fact_tracks t
JOIN dim_labels l ON t.label_id = l.label_id
GROUP BY l.label_name
HAVING total_tracks >= 100
ORDER BY hit_ratio_pct DESC, total_streams DESC
LIMIT 10;
"""

df_label_performance = pd.read_sql_query(query_2, conn)
df_label_performance

,label_name,total_tracks,hit_tracks,avg_popularity,total_streams,hit_ratio_pct
0,Independent,10767,602,48.3,2.383670e+09,5.59
1,Columbia,10690,597,48.1,2.162425e+09,5.58
2,Sony Music,10631,585,48.3,2.420776e+09,5.50
3,EMI,10574,581,48.3,2.451411e+09,5.49
4,Universal Music,10505,570,48.2,2.276478e+09,5.43
5,XL Recordings,10687,572,48.2,2.103665e+09,5.35
6,Island Records,10527,555,48.0,2.179514e+09,5.27
7,Warner Music,10619,546,47.9,2.242209e+09,5.14


## 4. Business Analysis 3: Year-over-Year (YoY) Release Trends & Acoustic Metrics

**Business Scenario:** Track the evolution of release volume, average danceability, and energy metrics over recent decades (2000 onwards).

In [13]:
query_3 = """
SELECT 
    release_year,
    COUNT(track_id) AS total_releases,
    ROUND(AVG(danceability), 3) AS avg_danceability,
    ROUND(AVG(energy), 3) AS avg_energy,
    ROUND(AVG(popularity), 2) AS avg_popularity
FROM fact_tracks
WHERE release_year >=2000
GROUP BY release_year
ORDER BY release_year DESC;
"""

df_yearly = pd.read_sql_query(query_3, conn)
df_yearly.head(10)

,release_year,total_releases,avg_danceability,avg_energy,avg_popularity
0,2025,7712,0.524,0.506,48.22
1,2024,7698,0.515,0.503,48.24
2,2023,7801,0.521,0.506,48.29
3,2022,7733,0.523,0.509,48.14
4,2021,7468,0.524,0.505,48.45
5,2020,7800,0.519,0.502,48.22
6,2019,7676,0.518,0.505,47.94
7,2018,7840,0.522,0.506,48.08
8,2017,7676,0.516,0.504,48.00
9,2016,7656,0.521,0.503,47.98


In [14]:
#Close the database connection to release file locks
conn.close()
print("Database connection successfully closed.")

Database connection successfully closed.
